# Тема 1. Тензоры и базовые операции
Тензор в PyTorch — это многомерный массив (как ndarray в NumPy), но с двумя ключевыми суперспособностями:
1) Он умеет производить вычисления на GPU (ускорение в десятки раз)
2) Он хранит историю операций для автоматического расчета градиентов (Autograd)


# Структура и архитектура PyTorch

PyTorch — это многослойная экосистема, состоящая из С++ ядра на нижнем уровне и удобных Python-модулей на верхнем.

---

## 1. Общая архитектура

### Основные компоненты ядра:
* **`torch.Tensor`** — Базовый многомерный массив. Поддерживает вычисления на GPU/TPU и хранит историю операций для расчета градиентов.
* **`torch.autograd`** — Движок автоматического дифференцирования. Строит динамический граф вычислений (Computational Graph) и вычисляет производные при вызове `.backward()`.
* **C++ Core (ATen & LibTorch)**:
  * **ATen**: Библиотека тензорных операций на C++.
  * **C10**: Нижний уровень абстракции для работы с памятью и устройствами.
  * **LibTorch**: C++ API для исполнения моделей без интерпретатора Python (для продакшена).

---

## 2. Ключевые модули Python API

### 🔹 `torch.nn` — Построение нейронных сетей
Содержит блоки для сборки архитектур:
* **`nn.Module`** — Базовый класс для всех моделей и слоев. Управляет параметрами (`.parameters()`) и переносом на устройства (`.to('cuda')`).
* **Слои**: `nn.Linear`, `nn.Conv2d`, `nn.LSTM`, `nn.Transformer` и др.
* **Функции потерь**: `nn.MSELoss`, `nn.CrossEntropyLoss` и др.

> **`torch.nn` vs `torch.nn.functional` (`F`)**:
> * `torch.nn` содержит слои со *состоянием* и обучемыми весами (объекты).
> * `torch.nn.functional` содержит *чистые функции* без состояния (например, `F.relu()`, `F.softmax()`).

---

### 🔹 `torch.optim` — Оптимизация
Модуль для обновления весов на основе градиентов:
* **Оптимизаторы**: `optim.SGD`, `optim.Adam`, `optim.AdamW`.
* **Планировщики (Schedulers)**: Управление изменением скорости обучения (Learning Rate).

---

### 🔹 `torch.utils.data` — Пайплайн данных
* **`Dataset`**: Абстрактный класс для определения логики получения единичного элемента (`__getitem__`) и размера датасета (`__len__`).
* **`DataLoader`**: Обертка над `Dataset` для батчевания, перемешивания (`shuffle`) и многопоточной загрузки (`num_workers`).

---

### 🔹 `torch.cuda` / `torch.mps` — Управление железом
Модули для контроля распределения памяти и взаимодействия с графическими ускорителями (NVIDIA CUDA, Apple Silicon MPS).

---

## 3. Сводная таблица модулей

| Модуль / Компонент | Назначение | Суть / Аналог |
| :--- | :--- | :--- |
| **`torch`** | Тензорные операции и математика | Аналог NumPy, но с поддержкой GPU |
| **`torch.nn`** | Слои, блоки и Loss-функции | ООП-конструктор нейросетей |
| **`torch.autograd`** | Расчет градиентов | Автоматическое дифференцирование |
| **`torch.optim`** | Оптимизаторы (Adam, SGD) | Алгоритмы обновления весов |
| **`torch.utils.data`** | `Dataset` и `DataLoader` | Загрузка и подготовка батчей |
| **`torch.jit`** | TorchScript | Экспорт и компиляция моделей |

---

## 4. Экосистема (Доменные библиотеки)

* **`torchvision`** — Компьютерное зрение (предобученные модели, аугментации, датасеты).
* **`torchaudio`** — Обработка аудио и спектрограмм.
* **`torchtext`** — Обработка текста и NLP.

---

## 5. Базовый цикл обучения (Training Loop)

```python
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Модель и оптимизатор
model = MyModel()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# 2. Цикл обучения
for inputs, targets in dataloader:
    optimizer.zero_grad()          # Обнуляем градиенты
    outputs = model(inputs)        # Прямой проход (Forward pass)
    loss = criterion(outputs, targets) # Расчет ошибки
    loss.backward()                # Обратный проход (Autograd)
    optimizer.step()               # Обновление весов

## 1. Перенос тензоров на GPU
Чтобы код работал везде (и на GPU, и на CPU), в PyTorch принять динамически опеределять устройство:

In [22]:
import torch

# Определяем устройство
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Используется устройство: {device}')
if device.type == 'cuda':
    print(f'Название GPU: {torch.cuda.get_device_name(0)}')

Используется устройство: cuda
Название GPU: NVIDIA GeForce RTX 5060 Ti


## 2. Способы создания тензоров

In [23]:
# Создание из Python-списка или NumPy массива
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])

# Тензоры со случайными значениями или нулями
zeros = torch.zeros(3, 3)
random_data = torch.randn(100, 784)  # 100 объектов, 784 признака

# Явное указание типа (float32 - стандарт для нейросетей)
x_float = torch.ones((2, 2), dtype=torch.float32)

print(random_data)

tensor([[-1.5378, -0.0376, -0.3864,  ..., -0.9039,  1.1666, -0.5004],
        [ 1.6049, -0.4895, -2.1728,  ..., -0.2949, -1.7660,  0.7313],
        [ 1.2130,  0.8125,  0.3383,  ..., -0.2105, -0.6096,  0.4336],
        ...,
        [ 1.8780, -0.5428,  0.3238,  ..., -0.5492,  0.4744, -0.2593],
        [ 0.4186,  0.7646,  0.0813,  ..., -0.6379,  0.7132,  0.1913],
        [ 0.9365, -0.0774, -0.8298,  ...,  0.8323,  1.1442,  1.8429]])


## 3. Перенос данных на устройство
Есть два основных способа отправить тензор на GPU:

In [24]:
# Способ 1: Использование .to(device) - самый универсальный и рекомендуемый!
data = torch.randn(64, 784)
data_gpu = data.to(device)

# Способ 2: Создание сразу на GPU
data_direct = torch.randn(64, 784, device=device)

print(f'Устройство тензора: {data_gpu.device}')

Устройство тензора: cuda:0


⚠️ Важное правило: Операции можно проводить только между тензорами, которые находятся на одном и том же устройстве! Если один тензор на CPU, а другой на GPU — PyTorch выдаст ошибку RuntimeError: Expected all tensors to be on the same device.

### 🎯 Задача Модуля 1: Нормализация и подготовка данных для GPU

**Контекст:** Вы получаете батч (пакет) изображений в виде сырого массива чисел с CPU, но для нейросети их нужно нормализовать, изменить форму и перенести на GPU.

**Задание:**
1. Определите устройство `device` (`cuda`, если GPU доступен, иначе `cpu`).
2. Создайте тензор `raw_images` размерности `(32, 28, 28)` (32 изображения 28x28) со случайными значениями от `0` до `255` типа `torch.float32` на CPU.
3. Нормализуйте значения тензора в диапазон от `0.0` до `1.0` (разделите на `255.0`).
4. Измените форму тензора на двумерный массив `(32, 784)` с помощью метода `.view()` или `.reshape()`.
5. Перенесите итоговый тензор на `device`.
6. Выведите на экран размерность (`.shape`), тип данных (`.dtype`) и устройство (`.device`) полученного тензора.

In [25]:
import torch


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
raw_images = torch.randint(low=0, high=256, size=(32, 28, 28), dtype=torch.float32, device='cpu')
raw_images_normalized = raw_images / 255
images = raw_images_normalized.reshape(32, 784)
images_gpu = images.to(device)
print(images_gpu.shape, images_gpu.dtype, images_gpu.device)

torch.Size([32, 784]) torch.float32 cuda:0


## Тема 2. Математические операции и Broadcasting
Нейросети на фундаментальном уровне — это комбинация матричных умножений, поэлементных операций и функций активации. Понимание того, как PyTorch работает с формами (shapes) и операциями над ними — ключ к эффективному коду.

### 1. Матричное умножение vs Поэлементное
В PyTorch важно не путать два вида умножения:

In [26]:
A = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
B = torch.tensor([[2.0, 0.0], [1.0, 2.0]])

# 1. Поэлементное умножение (Element-wise) — знаки * или torch.mul()
elem_wise = A * B
# Результат: [[1*2, 2*0], [3*1, 4*2]] -> [[2, 0], [3, 8]]

# 2. Матричное умножение (Matrix Multiplication) — оператор @ или torch.matmul()
matmul = A @ B
# Стандартное перемножение "строка на столбец"

⚠️ Главное правило матричного умножения: Внутренние размерности должны совпадать. Если $A$ имеет форму $(M, K)$, а $B$ имеет форму $(K, N)$, то результат $A @ B$ будет иметь форму $(M, N)$.

### 2. Broadcasting (Трансляция размерностей)
Broadcasting — это механизм, который позволяет PyTorch выполнять операции над тензорами разной размерности без явного копирования данных в памяти.

PyTorch автоматически "растягивает" меньший тензор вдоль недостающих осей по следующим правилам:

1) Размерности сравниваются справа налево.
2) Две размерности совместимы, если они равны, или одна из них равна 1.

Пример из реальной жизни: Прибавление вектора сдвигов (bias) к батчу признаков:

In [27]:
# Батч из 32 объектов, у каждого 64 признака
X = torch.randn(32, 64)

# Вектор смещений (bias) для 64 признаков
b = torch.randn(64)

# PyTorch сам растянет 'b' из формы (64,) до (1, 64),
# а затем применит к каждой из 32 строк!
result = X + b  # Форма результата: (32, 64)

### 🎯 Задача Модуля 2: Эмуляция линейного слоя с байасом и нормализацией

**Контекст:** Вы пишете свой первый полносвязный слой "вручную" через базовую алгебру PyTorch, используя матричное умножение, broadcasting и агрегацию.

**Задание:**
1. Задайте `device` (`cuda`, если доступен, иначе `cpu`).
2. Создайте тензор входов `X` размерности `(64, 100)` (батч из 64 объектов, 100 признаков) со случайными нормальными числами (`torch.randn`), сразу на `device`.
3. Создайте матрицу весов `W` размерности `(100, 10)` и вектор сдвигов (bias) `b` размерности `(10)` с помощью `torch.randn` на `device`.
4. Выполните прямое прохождение полносвязного слоя: $Y = X \cdot W + b$. *(Используйте матричное умножение и broadcasting!)*
5. Посчитайте вектор средних значений полученных выходов по каждому из 10 признаков вдоль батча (получится вектор формы `(10,)`).
6. Вычтите этот вектор средних из каждой строки тензора $Y$ (центрирование данных с помощью broadcasting).
7. Выведите форму (`.shape`) и устройство (`.device`) итогового тензора.

In [28]:
import torch


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
X = torch.randn(size=(64, 100), device=device)
W = torch.randn(100, 10, device=device)
b = torch.randn(10, device=device)
Y = X @ W + b
means = torch.mean(Y, dim=0, keepdim=False)
Y -= means
print(Y.shape, Y.device)

torch.Size([64, 10]) cuda:0


## Модуль 3. Автоматическое дифференцирование (Autograd)
В основе обучения любых нейросетей лежит метод обратного распространения ошибки (Backpropagation). Чтобы не вычислять производные сложнейнших функций вручную, в PyTorch встроена система Autograd.

Когда ты выполняешь операции над тензорами, PyTorch на лету строит направленный ациклический граф вычислений. При вызове метода .backward() PyTorch автоматически проходит по этому графу назад и с помощью цепного правила считает градиенты всех параметров.

### 1. Флаг requires_grad=True
По умолчанию тензоры создаются с requires_grad=False. Если мы хотим, чтобы PyTorch отслеживал операции над тезорами и считал по нему производные (кк для весов модели), нужно указать requires_grad=True:

In [29]:
import torch


# Вес модели, по которому мы захотим посчитать градиент
w = torch.tensor([2.0], requires_grad=True)
b = torch.tensor([1.0], requires_grad=True)

# Входные данные (по ним градиент обычно не нужен)
x = torch.tensor([3.0])

# Прямой проход (Forward Pass): y = w * x + b
y = w * x + b

### Запуск .backward() и получение градиентов
Чтобы запустить вычисление градиентов, нужно вызывать метод .backward() у итоговой скалярной величины (обычно это функция потерь / Loss):

In [30]:
# Посчитаем простую функцию потерь: loss = y^2
loss = y ** 2

# Запускаем обратный проход (Backpropogation)
loss.backward()

# Градиенты d(loss)/dw и d(loss)/db автоматически сохранились в .grad
print(f"Градиент w: {w.grad}")  # d(loss)/dw = 2 * y * x = 2 * 7 * 3 = 42.0
print(f"Градиент b: {b.grad}")  # d(loss)/db = 2 * y * 1 = 2 * 7 * 1 = 14.0


Градиент w: tensor([42.])
Градиент b: tensor([14.])


### 3. Важные особенности Autograd
1) Накопление градиентов (Gradient Accumulation):
При повторном вызове .backward() градиенты складываются с предыдущими значениями! В реальном обучении перед каждым шагом оптимизации их зануляют: `w.grad.zero_()`.

2) Отключение градиентов (`torch.no_grad()`):
Во время валидации или тестирования модели нам не нужен граф вычислений — это экономит кучу памяти GPU и ускоряет работу:

In [31]:
with torch.no_grad():
    y_pred = w * x + b  # Операции внутри не строят граф вычислений

3) Отсоединение от графа (`.detach()`):
Создает новый тензор с теми же данными, но "отрезанный" от истории графа (у нового тензора `requires_grad=False`).

### 🎯 Задача Модуля 3: Ручной шаг градиентного спуска (Gradient Descent Step)

**Контекст:** Вы пишете базовый цикл обновления параметров нейросети «с нуля» без использования готовых оптимизаторов, с чистым Autograd и ручной очисткой градиентов.

**Задание:**
1. Задайте `device` (`cuda`, если доступен, иначе `cpu`).
2. Создайте обучаемый параметр `w` (скаляр или тензор формы `(1,)`) со значением `2.0`, указав `requires_grad=True` и отправив его на `device`.
3. Задайте константу скорости обучения `lr = 0.1` (learning rate).
4. Выполните 1-ю итерацию:
   - Вычислите функцию: $f(w) = w^2 - 4w + 4$.
   - Вызовите `.backward()` для вычисления производной $\frac{df}{dw}$.
   - Сохраните значение градиента `w.grad.item()` в переменную `grad1`.
   - Обновите параметр `w` вручную по формуле $w = w - lr \times \frac{df}{dw}$. *(💡 Внимание: обновление `w` делайте внутри контекста `with torch.no_grad():`, чтобы операция обновления не попала в граф вычислений!)*
   - Занулите градиент у `w` с помощью `w.grad.zero_()`.
5. Выполните 2-ю итерацию:
   - Снова вычислите функцию $f(w) = w^2 - 4w + 4$ с уже обновленным `w`.
   - Запустите `.backward()`.
   - Сохраните новый градиент `w.grad.item()` в переменную `grad2`.
6. Выведите на экран:
   - Значение `grad1` и обновленное `w` после 1-й итерации.
   - Значение `grad2` и значение функции $f(w)$ после 2-й итерации.

In [32]:
import torch


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
w = torch.tensor([2.0], requires_grad=True, device=device)
lr = 0.1

f = w ** 2 - 4 * w + 4
f.backward()
grad1 = w.grad.item()
with torch.no_grad():
    w -= lr * grad1
w.grad.zero_()

f = w ** 2 - 4 * w + 4
f.backward()
grad2 = w.grad.item()

print(grad1, w)
print(grad2, f)

0.0 tensor([2.], device='cuda:0', requires_grad=True)
0.0 tensor([0.], device='cuda:0', grad_fn=<AddBackward0>)


## Модуль 4: Построение нейросетей через `torch.nn`
В предыдущих модулях мы делали все вручную через сырые тензоры. Но при построении глубоких сетей ручная декларация матриц весов и байасов быстро превращается в хаос.

Модуль `torch.nn` предоставляет высокоуровневые строительные блоки: слои, функции активации, блоки регуляризации и удобные контейнеры.

### 1. Полносвязный слой (nn.Linear)
Слой `nn.Linear(in_features, out_features)` берет на себя создание матрицы весов `W` и вектора смещений `b`, а также автоматическую случайную инициализацию:

In [33]:
import torch
import torch.nn as nn


# Создаем слой: 784 входов (признаков), 64 выхода (скрытых нейрона)
linear_layer = nn.Linear(in_features=784, out_features=64)

# У слоя сразу есть собственные параметры (веса и байасы):
print(linear_layer.weight.shape)
print(linear_layer.bias.shape)

torch.Size([64, 784])
torch.Size([64])


### 2. Функции активации и Dropout
Для введения нелинейности используются функции активации (`nn.ReLU`, `nn.Sigmoid`, `nn.GELU`), а для защиты от переобучения - слой регуляризации `nn.Dropout(p)`:
* `nn.ReLU()`: Обнуляет все отрицательные значения: $f(x) = \max(0, x)$.
* `nn.Dropout(p=0.2)`: Во время обучения случайным образом с вероятностью $p$ "выключает" (зануляет) часть нейронов.

### 3. Контейнер `nn.Sequential`
`nn.Sequential` - это самый простой способ собрать цепь из слоев, через которые данные будут проходить последовательно друг за другом:

In [34]:
# Собираем простую сеть из 2-х слоев:
model = nn.Sequential(
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Dropout(p=0.1),
    nn.Linear(128, 10) # 10 классов на выходе
)

# Переносим ВСЮ сетьна GPU в одну строчку!
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Прогоняем батч данных через сеть
x = torch.randn(32, 784, device=device)
output = model(x) # Вызывает метод forward внутри Sequential
print(output.shape)

torch.Size([32, 10])


💡 Как устроен `add_module`?

В `nn.Sequential` слои можно добавлять не только разом в конструктор, но и по очереди через метод `.add_module(name, module)`. Имя модуля задается строкой, например:

In [35]:
seq = nn.Sequential()
seq.add_module('fc1', nn.Linear(784, 64))
seq.add_module('relu1', nn.ReLU())

### 🎯 Задача Модуля 4: Динамическое построение сети через `nn.Sequential` и `add_module`

**Контекст:** Вы подготавливаетесь к написанию кастомного класса `Perceptron`. Вам нужно научиться динамически в цикле собирать цепочку из слоев `Linear`, `ReLU` и `Dropout` с помощью метода `add_module`.

**Задание:**
1. Задайте `device` (`cuda`, если доступен, иначе `cpu`).
2. Задайте переменные параметров:
   - `input_dim = 100`
   - `hidden_dim = 32`
   - `output_dim = 2`
   - `num_layers = 3` (количество скрытых слоев)
   - `p = 0.2` (вероятность dropout)
3. Создайте пустой контейнер `model = nn.Sequential()`.
4. С помощью цикла `for i in range(num_layers)` динамически добавьте в `model` с помощью метода `.add_module()`:
   - Линейный слой `torch.nn.Linear(prev_size, hidden_dim)` с именем `'layer{}'.format(i)`
   - Активацию `torch.nn.ReLU()` с именем `'relu{}'.format(i)`
   - Дропаут `torch.nn.Dropout(p=p)` с именем `'dropout{}'.format(i)`
   - *(Не забудьте обновлять переменную `prev_size` на каждом шаге цикла!)*
5. После окончания цикла добавьте финальный классификационный слой `torch.nn.Linear(prev_size, output_dim)` с именем `'classifier'`.
6. Перенесите `model` на `device`.
7. Создайте случайный батч данных `X` формы `(16, input_dim)` на `device`.
8. Пропустите `X` через модель, получите `output` и выведите на экран:
   - Саму модель `print(model)` (PyTorch красиво напечатает структуру ее слоев).
   - Форму (`.shape`) и устройство (`.device`) полученного `output`.

In [36]:
import torch
import torch.nn as nn


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

input_dim = 100
hidden_dim = 32
output_dim = 2
num_layers = 3
p = 0.2

model = nn.Sequential()
prev_size = input_dim
for i in range(num_layers):
    model.add_module(f'layer{i+1}', nn.Linear(prev_size, hidden_dim))
    model.add_module(f'relu{i+1}', nn.ReLU())
    model.add_module(f'dropout{i+1}', nn.Dropout(p=p))
    prev_size = hidden_dim

model.add_module('classifier', nn.Linear(prev_size, output_dim))
model = model.to(device)

X = torch.randn(16, input_dim, device=device)
output = model(X)
print(model)
print(output.shape, output.device)

Sequential(
  (layer1): Linear(in_features=100, out_features=32, bias=True)
  (relu1): ReLU()
  (dropout1): Dropout(p=0.2, inplace=False)
  (layer2): Linear(in_features=32, out_features=32, bias=True)
  (relu2): ReLU()
  (dropout2): Dropout(p=0.2, inplace=False)
  (layer3): Linear(in_features=32, out_features=32, bias=True)
  (relu3): ReLU()
  (dropout3): Dropout(p=0.2, inplace=False)
  (classifier): Linear(in_features=32, out_features=2, bias=True)
)
torch.Size([16, 2]) cuda:0


## Модуль 5. Кастомные модули (nn.Module) и продвинутые фишки
Теперь мы объединим абсолютно все знания, полученные за курс, чтобы разобрать наш целевой класс Perceptron до каждого символа.

### 1. Как устроен кастомный класс на базе nn.Module
Все нейросети в PyTorch наследуются от класса torch.nn.Module. У него есть два главных требования к структуре:
1) super().__init__() в конструкторе: ОБЯЗАТЕЛЬНО должен вызываться первым делом. Он инициализирует внутренюю инфраструктуру PyTorch (регистрацию слоев, параметров, буферов)
2) Метод forward(self, input): определяет, как данные проходят через слои. Когда ты пишешь model(X), PyTorch под капотом автоматически вызывает именно метод forward.

### 2. Продвинутый трюк: свойство @property def device


In [37]:
@property
def device(self):
    for p in self.parameters():
        return p.device

#### Зачем это нужно?
У самого класса torch.nn.Module по умолчанию нет прямого атрибута model.device. Если ты перенесешь модель на GPU через model.to(device), слои перейдут на GPU, но узнать устройство модели напрямую через model.device нельзя (вызовет AttributeError)

Этого геттер-свойство решает эту проблему:
1) self.parameters() возвращает генератор всех весов (nn.Parameter) модели.
2) for p in self.parameters(): return p.device берет самый первый вес сети и возврщает его .device.
3) Декоратор @property позволяет обращаться к этому методу как к обычному полю - model.device вместо model.device()

⚠️ Маленький нюанс: Если у модели нет вообще ни одного слоя с весами (что бывает крайне редко), генератор будет пуст. В реальном коде иногда добавляют дефолтный return torch.device('cpu') на случай отсутствия параметров.

### 🎯 Задача Модуля 5: Реализация и тестирование кастомного класса `Perceptron`

**Контекст:** Вы пишете финальный класс `Perceptron`, проверяете работу динамической генерации слоев, кастомного свойства `.device` и корректность переноса всей модели и данных на GPU.

**Задание:**
1. Определите `device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')`.
2. Напишите полный класс `Perceptron(nn.Module)` точно с такой же логикой, как описано выше (включая свойство `@property def device`, динамическое добавление слоев в `__init__` и метод `forward`).
3. Создайте экземпляр класса `model` со следующими гиперпараметрами:
   - `input_dim = 784`
   - `num_layers = 2`
   - `hidden_dim = 128`
   - `output_dim = 10`
   - `p = 0.1`
4. Перенесите `model` на `device` с помощью `model.to(device)`.
5. Проверьте работу свойства `.device`: выведите на экран `model.device`.
6. Создайте батч входных данных `X` формы `(64, 784)` со случайными значениями, автоматически поместив его на `model.device` *(💡 подсказка: используйте `device=model.device` в `torch.randn`!)*.
7. Пропустите `X` через модель `output = model(X)`.
8. Выведите на экран:
   - Форму (`.shape`) полученного `output`.
   - Устройство (`.device`), на котором находится `output`.

In [38]:
import torch
import torch.nn as nn


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class Perceptron(nn.Module):

    @property
    def device(self):
        for p in self.parameters():
            return p.device


    def __init__(self, input_dim=784, num_layers=0, hidden_dim=64, output_dim=10, p=0.0):
        super().__init__()

        self.layers = nn.Sequential()
        prev_size = input_dim
        for i in range(num_layers):
            self.layers.add_module(f'layer{i}', nn.Linear(prev_size, hidden_dim))
            self.layers.add_module(f'relu{i}', nn.ReLU())
            self.layers.add_module(f'dropout{i}', nn.Dropout(p=p))
            prev_size = hidden_dim

        self.layers.add_module('classifier', nn.Linear(prev_size, output_dim))

    def forward(self, X):
        return self.layers(X)

model = Perceptron(input_dim=784, num_layers=2, hidden_dim=128, output_dim=10, p=0.1)
model = model.to(device)
X = torch.randn(64, 784, device=model.device)
output = model(X)

print(output.shape, output.device)

torch.Size([64, 10]) cuda:0


💡 Архитектурный факт: В PyTorch вообще ВСЕ модели строится по принципу «матрешки». Выходной класс модели — это внешняя обертка, внутри которой могут лежать другие обертки (nn.Sequential или кастомные блоки), а внутри них — базовые слои вроде nn.Linear.

# Программа второго уровня

## Модуль 6: Загрузка данных (Dataset и DataLoader)
При работе с реальными данными (картинки, тексты, звуки) мы не сможем загрузить весь датасет в оперативную память или VRAM GPU разом - памяти просто не хватит.

Для этого в PyTorch есть два ключевых класса:
1. Dataset - отвечает за доступ к одному конкретного объекту данных по индексу
2. DataLoader - оборачивает Dataset, автоматически нарезает данные на батчи (пакеты), перемешивает их и загружает в фоновых процессах.

In [39]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# 1. Создаем симуляцию датасета (1000 объектов, 784 признака) и меток классов (1000 меток)
X_data = torch.randn(1000, 784)
y_data = torch.randint(0, 10, size=(1000,)) # Классы от 0 до 9

# 2. Упаковываем тензоры в простой TensorDataset
dataset = TensorDataset(X_data, y_data)

# 3. Создаем DataLoader
dataloader = DataLoader(
    dataset=dataset,
    batch_size=32,    # Размер пакета (батча)
    shuffle=True,    # Перемешивать данные каждую эпоху (важно для обучения!)
    drop_last=False    # Оставлять ли последний неполный батч
)

# 4. Итерация по батчам в цикле обучения
for batch_X, batch_y in dataloader:
    # batch_X имеет форму (32, 784)
    # batch_y имеет форму (32,)
    pass

💡 Важное правило пересылки на GPU:

Данные в `DataLoader` хранятся на CPU. На GPU их следует отправлять внутри цикла обучения для каждого батча отдельно (`batch_X = batch_X.to(device)`), чтобы не перегружать VRAM GPU.

### 🎯 Задача Модуля 6: Создание пайплайна данных и проверка итератора

**Контекст:** Вы подготавливаете данные для обучения модели `Perceptron`. Вам нужно обернуть сырые тензоры в `TensorDataset`, настроить `DataLoader` и проверить размеры батчей при итерации.

**Задание:**
1. Задайте `device` (`cuda`, если доступен, иначе `cpu`).
2. Создайте синтетические данные на CPU:
   - `X_raw` формы `(500, 784)` со случайными нормальными числами (`torch.randn`).
   - `y_raw` формы `(500,)` с целыми случайными числами от `0` до `9` (`torch.randint`).
3. Создайте `dataset = TensorDataset(X_raw, y_raw)`.
4. Создайте `loader = DataLoader(dataset, batch_size=64, shuffle=True)`.
5. Выведите на экран общее количество батчей в `loader` с помощью `len(loader)`.
6. Достаньте **первый батч** из `loader` с помощью `next(iter(loader))`.
7. Перенесите полученные `batch_X` и `batch_y` на `device`.
8. Выведите на экран:
   - Форму (`.shape`) и устройство (`.device`) для `batch_X`.
   - Форму (`.shape`) и устройство (`.device`) для `batch_y`.

In [40]:
import torch
from torch.utils.data import TensorDataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

X_raw = torch.randn(500, 784, device='cpu')
y_raw = torch.randint(low=0, high=10, size=(500,), device='cpu')

dataset = TensorDataset(X_raw, y_raw)

loader = DataLoader(dataset=dataset, batch_size=64, shuffle=True)
print(len(loader))
first_batch = next(iter(loader))
batch_X = first_batch[0].to(device)
batch_y = first_batch[1].to(device)

print(batch_X.shape, batch_X.device)
print(batch_y.shape, batch_y.device)

8
torch.Size([64, 784]) cuda:0
torch.Size([64]) cuda:0


## Модуль 7. Функции потерь (Loss Functions) и Оптимазаторы
Мы научились собирать сеть (Perceptron), подготавливать данные (DataLoader) и переносить все на GPU. Теперь нужно научить сеть учиться. За это отвечают два компонента:
1) Функция потерь (Loss Function): Измеряет, насколько сильно предсказания модели отличаются от реальных ответов
2) Опитимизатор (Optimizer): Алгоритм, который на основе посчитанных градиентов обновляет веса модели, чтобы уменьшить ошибку.

### 1. Функция потерь для классификации (nn.CrossEntropyLoss)
Для задач многоклассовой классификации (например, определение цифры от 0 до 9) стандартом является Cross-Entropy Loss (перекрестная энтропия).

В PyTorch есть ключевая особенность nn.CrossEntropyLoss:
* Она принимает на вход сырые выходы модели (Logits) — то есть значения с последнего линейного слоя БЕЗ примененного Softmax.
* Под капотом она сама эффективнее и численно стабильнее применяет LogSoftmax и вычисляет ошибку.

In [41]:
import torch
import torch.nn as nn


criterion = nn.CrossEntropyLoss()

logits = torch.tensor([[2.0, 1.0, 0.1],
                       [0.5, 3.0, 0.2]])

targets = torch.tensor([0, 1]) # Для первого объекта класс 0, для второго — 1

loss = criterion(logits, targets)
print(f"Значение ошибки: {loss.item():.4f}")

Значение ошибки: 0.2753


### 2. Оптимизаторы (`torch.optim`)
Оптимизатор берет параметры модели (model.parameters()) и управляет их обновлением. Самый популярный и универсальный оптимизатор — Adam (или его улучшенная версия AdamW).

Работа с оптимизатором всегда состоит из 3 обязательных шагов:

In [42]:
import torch.optim as optim

# Инициализируем оптимизатор, передавая веса модели и learning_rate (lr)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# --- ВНУТРИ ШАГА ОБУЧЕНИЯ ---

# Шаг 1: Зануляем градиенты с прошлого шага!
optimizer.zero_grad()

# Шаг 2: Считаем новые градиенты через обратный проход
loss.backward()

# Шаг 3: Обновляем веса модели: W_new = W_old - lr * grad
optimizer.step()

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

⚠️ Почему важен `optimizer.zero_grad()`?

Помнишь Модуль 3? PyTorch по умолчанию складывает (накапливает) градиенты при каждом вызове `.backward()`. Если перед новым батчем не вызвать `optimizer.zero_grad()`, градиенты от предыдущего батча прибавятся к текущим, и обучение быстро «взорвется».

### 🎯 Задача Модуля 7: Ручная симуляция одного шага обучения (Training Step)

**Контекст:** Вы объединяете модель `Perceptron`, данные, функцию потерь `nn.CrossEntropyLoss` и оптимизатор `Adam`, чтобы выполнить один полный изолированный шаг обновления весов сети.

**Задание:**
1. Переиспользуйте или объявите класс `Perceptron` из Модуля 5.
2. Определите `device` (`cuda`, если доступен, иначе `cpu`).
3. Создайте модель `model = Perceptron(input_dim=784, num_layers=1, hidden_dim=64, output_dim=10)`. Перенесите её на `device`.
4. Создайте функцию потерь `criterion = nn.CrossEntropyLoss()` и оптимизатор `optimizer = torch.optim.Adam(model.parameters(), lr=0.01)`.
5. Создайте фейковый батч данных прямо на `device`:
   - `batch_X` формы `(32, 784)` с помощью `torch.randn`.
   - `batch_y` формы `(32,)` с помощью `torch.randint` (значения от 0 до 9).
6. Запомните значение весов первого слоя **ДО** обновления:
   `weight_before = model.layers.layer0.weight.clone()`
7. Выполните шаг обучения:
   - Сбросьте градиенты через `optimizer.zero_grad()`.
   - Получите предсказания `outputs = model(batch_X)`.
   - Посчитайте ошибку `loss = criterion(outputs, batch_y)`.
   - Запустите `loss.backward()`.
   - Сделайте шаг оптимизатора `optimizer.step()`.
8. Выведите на экран:
   - Значение ошибки `loss.item()`.
   - Проверку того, изменились ли веса после шага оптимизатора с помощью функции `torch.equal(weight_before, model.layers.layer0.weight)` (должно вернуть `False`, так как веса изменились!).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class Perceptron(nn.Module):

    @property
    def device(self):
        for p in self.parameters():
            return p.device


    def __init__(self, input_dim=784, num_layers=0, hidden_dim=64, output_dim=10, p=0.0):
        super().__init__()

        self.layers = nn.Sequential()
        prev_size = input_dim
        for i in range(num_layers):
            self.layers.add_module(f'layer{i}', nn.Linear(prev_size, hidden_dim))
            self.layers.add_module(f'relu{i}', nn.ReLU())
            self.layers.add_module(f'dropout{i}', nn.Dropout(p=p))
            prev_size = hidden_dim

        self.layers.add_module('classifier', nn.Linear(prev_size, output_dim))

    def forward(self, X):
        return self.layers(X)


model = Perceptron(input_dim=784, num_layers=1, hidden_dim=64, output_dim=10).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

batch_X = torch.randn(32, 784, device=device)
batch_y = torch.randint(0, 10, size=(32,), device=device)

weight_before = model.layers.layer0.weight.clone()

optimizer.zero_grad()
outputs = model(batch_X)
loss = criterion(outputs, batch_y)
loss.backward()
optimizer.step()

print(loss.item())
print(torch.equal(weight_before, model.layers.layer0.weight))

## Модуль 8. Полный цикл обучения (Training $ Evaluation Loop)
Теперь у нас есть абсолютно все компоненты, чтобы собрать полный, настоящий цикл обучения модели на несколько эпох с подсчетом accuracy на обучении и валидации.


### 1. Переключение режимов: `model.train()` vs `model.eval()`
У некоторых слоев в PyTorch (например, nn.Dropout и nn.BatchNorm) поведение кардинально различается во время обучения и во время тестирования:
* model.train(): переводит все слои в режим обучения. Dropout случайным образом зануляет часть нейронов
* model.eval(): переводит все слои в режим оценки (валидации/теста). Dropout выключается и пропускает через себя 100% сигналов.


💡 Правило: Всегда явно вызывай model.train() перед циклом обучения по эпохе и model.eval() перед блоком валидации/теста!

### 2. Расчет точности (Accuracy)
Для многоклассовой классификации предсказанным классом является индекс с максимальным логитом (выходом сети).


In [ ]:
# outputs имеет форму (32, 10)
# torch.max(outputs, dim=1) возвращает (максимальные_значения, индексы_максимумов)
_, preds = torch.max(outputs, dim=1)

# Считаем количество совпадений с реальными метками batch_y
correct_count = (preds == batch_y).sum().item()
accuracy = correct_count / batch_y.size(0) # Доля правильных ответов

### 3. Анатомия полного цикла (2 вложенных цикла)
Структура любого пайплайна в PyTorch всегда выглядит так:

In [ ]:
epochs = 5

for epoch in range(epochs):
    # --- 1. ФАЗА ОБУЧЕНИЯ ---
    model.train()
    train_loss = 0.0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # --- 2. ФАЗА ВАЛИДАЦИИ / ОЦЕНКИ ---
    model.eval()
    val_loss = 0.0

    with torch.no_grad(): # Отключаем граф вычислений для экономии VRAM
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item()

    print(f"Epoch {epoch+1} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss/len(val_loader):.4f}")

### 🎯 Задача Модуля 8 (Реальные данные): Обучение Perceptron на датасете MNIST

**Контекст:** Вы обучаете и тестируете свою модель `Perceptron` на настоящем классическом датасете MNIST, отслеживая рост точности (Accuracy) от эпохи к эпохе.

**Задание:**
1. Задайте `device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')`.
2. Загрузите датасеты `train_dataset` и `test_dataset` MNIST с помощью `torchvision.datasets.MNIST` с трансформацией `transforms.ToTensor()`.
3. Оберните их в `DataLoader`:
   - `train_loader` с `batch_size=64`, `shuffle=True`.
   - `test_loader` с `batch_size=64`, `shuffle=False`.
4. Создайте экземпляр вашей модели:
   `model = Perceptron(input_dim=784, num_layers=2, hidden_dim=128, output_dim=10, p=0.1)`. Перенесите её на `device`.
5. Задайте `criterion = nn.CrossEntropyLoss()` и `optimizer = optim.Adam(model.parameters(), lr=0.001)`.
6. Напишите цикл обучения на `epochs = 3`:
   - **В фазе обучения (`model.train()`):**
     * Проходите по `train_loader`.
     * Переносите батчи на `device`.
     * Выравнивайте `batch_X` из `(batch_size, 1, 28, 28)` в `(batch_size, 784)` через `.view()` или `.reshape()`.
     * Делайте `zero_grad()`, `forward`, вычисляйте `loss`, `.backward()` и `optimizer.step()`.
     * Накапливайте сумму потерь и количество совпавших предсказаний (`preds == batch_y`).
   - **В фазе валидации (`model.eval()` + `with torch.no_grad():`):**
     * Проходите по `test_loader`.
     * Выполняйте `flatten` для `batch_X`, получайте выходы модели и накапливайте точность и потери.
7. В конце каждой эпохи выводите на экран показатели:
   `Epoch [X/3] | Train Loss: X.XXXX | Train Acc: XX.XX% | Test Loss: X.XXXX | Test Acc: XX.XX%`

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

## -----ПОДГОТОВКА ДАННЫХ-----
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])


train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=64,
    shuffle=True
)
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=64,
    shuffle=True
)

In [ ]:
# -----МОДЕЛЬ-----

class Perceptron(nn.Module):
    @property
    def device(self):
        for p in self.parameters():
            return p.device

    def __init__(self, input_dim=784, num_layers=2, hidden_dim=128, output_dim=10, p=0.1):
        super().__init__()
        self.layers = nn.Sequential()
        prev_size = input_dim
        for i in range(num_layers):
            self.layers.add_module(f'layer{i+1}', nn.Linear(prev_size, hidden_dim))
            self.layers.add_module(f'relu{i+1}', nn.ReLU())
            self.layers.add_module(f'dropout{i+1}', nn.Dropout(p=p))
            prev_size = hidden_dim
        self.layers.add_module(f'classifier', nn.Linear(prev_size, output_dim))

    def forward(self, X):
        return self.layers(X)


model = Perceptron(input_dim=784, num_layers=2, hidden_dim=128, output_dim=10).to(device)


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# -----ОБУЧЕНИЕ-----

epochs = 3
for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    train_correct = 0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.reshape(-1, 784).to(device), batch_y.to(device)

        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, preds = torch.max(outputs, dim=1)
        train_correct += (preds == batch_y).sum().item()

    total_train_samples = len(train_loader.dataset)
    avg_train_loss = train_loss / len(train_loader)
    train_acc = (train_correct / total_train_samples) * 100

    # -----ВАЛИДАЦИЯ-----

    model.eval()
    val_loss = 0.0
    val_correct = 0

    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.reshape(-1, 784).to(device), batch_y.to(device)
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item()

            _, preds = torch.max(outputs, dim=1)
            val_correct += (preds == batch_y).sum().item()

    total_val_samples = len(test_loader.dataset)
    avg_val_loss = val_loss / len(test_loader)
    val_acc = (val_correct / total_val_samples) * 100


    print(f"Epoch {epoch + 1}/{epochs} | "
          f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}%")
